# 🏂 Zero to Snowflake ハンズオン

このノートブックは、Snowflake を初めて触れる方向けの **ステップバイステップ ハンズオン** です。  
CitiBike（ニューヨークの自転車シェアサービス）のデータを使い、Snowflake の主要機能を体験します。

## 📋 ハンズオンの流れ

| ステップ | 内容 |
|--------|------|
| Step 3 | データロードの準備（DB・スキーマ作成） |
| Step 4 | テーブル・ステージ・ファイルフォーマット作成 |
| Step 5 | データロード ＆ スケールアップ体験 |
| Step 6 | クエリ・キャッシュ体験 |
| Step 7 | 半構造化データ（JSON）の操作 |
| Step 8 | タイムトラベル・ゼロコピークローン |
| Step 9 | ロール管理（RBAC） |
| Step 11 | 環境のリセット |

---
> ⚠️ **注意事項**
> - 各セルは **順番に実行** してください。
> - このハンズオンでは `SYSADMIN` および `ACCOUNTADMIN` ロールを使用します。
> - ウェアハウス名は `compute_wh` を使用します（環境によって異なる場合は適宜変更してください）。

## 🔌 接続情報の確認

現在接続しているアカウント・ロール・ウェアハウスを確認します。

In [ ]:
%%sql -r dataframe_1
-- ■ 現在の接続情報を確認する
-- Snowflake では「どのアカウントに」「どのロールで」「どのウェアハウスを使って」
-- 作業しているかを常に把握することが大切です。
SELECT
    CURRENT_ACCOUNT()   AS account,    -- 接続しているアカウント名
    CURRENT_ROLE()      AS role,       -- 現在使用しているロール（権限セット）
    CURRENT_WAREHOUSE() AS warehouse;  -- 現在使用しているウェアハウス（計算リソース）


---
## Step 3｜データロードの準備

> 💡 **Snowflake Tips: オブジェクトの階層構造**
>
> Snowflake のオブジェクトは以下の階層で管理されます:
>
> ```
> アカウント（Account）
>   └── データベース（Database）
>         └── スキーマ（Schema）
>               └── テーブル / ビュー / ステージ / ファイルフォーマット ...
> ```
>
> - **Database**: データのコンテナ。論理的なグループ単位。
> - **Schema**: テーブルなどオブジェクトのコンテナ。データベース内の区画。
> - **Warehouse**: クエリ実行の **コンピューティング** リソース（≠ストレージ）。
> - Snowflake ではストレージとコンピューティングが **完全に分離** されています。

ロールを `SYSADMIN` に切り替え、CitiBike データベースを作成します。

In [ ]:
%%sql -r dataframe_2
-- ■ ロールを SYSADMIN に切り替える
-- SYSADMIN はデータベースやテーブルなどのオブジェクトを作成・管理するロールです。
USE ROLE sysadmin;

-- ■ CitiBike データベースを作成する
-- OR REPLACE を付けると既存の場合でも上書き作成できます（べき等性の確保）。
CREATE OR REPLACE DATABASE citibike;

-- ■ 作業対象のデータベース・スキーマ・ウェアハウスを指定する
-- Snowflake ではクエリを実行する前に「どこで・何を使って」作業するかを宣言します。
USE DATABASE citibike;    -- 操作するデータベースを指定
USE SCHEMA public;        -- 操作するスキーマを指定（デフォルトは public）
USE WAREHOUSE compute_wh; -- クエリ実行に使うウェアハウス（計算リソース）を指定


---
## Step 4｜データ読み込みの準備

> 💡 **Snowflake Tips: ステージ（Stage）とは？**
>
> **ステージ** は、データを Snowflake のテーブルにロードする前の **一時置き場** です。
>
> | 種類 | 説明 |
> |------|------|
> | 外部ステージ | AWS S3 / Azure Blob / GCS などのクラウドストレージを指定 |
> | 内部ステージ | Snowflake が管理するストレージ |
>
> **ファイルフォーマット（File Format）** は、ロードするデータの形式（CSV、JSON、Parquet 等）と  
> オプション（区切り文字、ヘッダースキップ、圧縮形式など）を定義するオブジェクトです。


### 4-1｜外部ステージを作成

AWS S3 上に置かれた CitiBike データを指す外部ステージを作成し、ファイル一覧を確認します。  
**S3 には 377 ファイル（gzip 圧縮後 1.9GB）、約 6,150 万行** のデータがあります。

In [ ]:
%%sql -r dataframe_3
-- ■ 外部ステージを作成する
-- ステージ = Snowflake がデータをロードする「参照先」の定義です。
-- ここでは AWS S3 上にある CitiBike データを指す外部ステージを作成します。
-- OR REPLACE で同名ステージがあれば上書きします。
CREATE OR REPLACE STAGE citibike_trips
    url = 's3://snowflake-workshop-lab/japan/citibike-trips/';
-- この S3 バケットには約 377 ファイル（gzip 圧縮後 1.9GB、約 6,150 万行）があります。


In [ ]:
%%sql -r dataframe_4
-- ■ ステージ内のファイル一覧を確認する
-- LIST コマンドでステージに含まれるファイルの名前・サイズ・更新日時を確認できます。
-- @ はステージを参照するときに付ける記号です。
LIST @citibike_trips;


### 4-2｜ファイルフォーマットを作成

CSV データを綺麗にロードするためのファイルフォーマットを定義します。

In [ ]:
%%sql -r dataframe_5
-- ■ CSV ファイルの読み込み形式（File Format）を定義する
-- File Format = 「このファイルをどのルールで解釈するか」を Snowflake に教えるオブジェクトです。
CREATE OR REPLACE FILE FORMAT csv
    TYPE = 'csv'                           -- ファイル種別: CSV
    COMPRESSION = 'auto'                   -- 圧縮形式: 自動判定（gzip など）
    FIELD_DELIMITER = ','                  -- 列の区切り文字: カンマ
    RECORD_DELIMITER = '\n'              -- 行の区切り文字: 改行
    SKIP_HEADER = 0                        -- ヘッダー行をスキップする行数（0 = スキップなし）
    FIELD_OPTIONALLY_ENCLOSED_BY = '\042' -- 文字列をダブルクォートで囲む場合の指定
    TRIM_SPACE = false                     -- 値前後の空白を除去しない
    ERROR_ON_COLUMN_COUNT_MISMATCH = false -- 列数が一致しなくてもエラーにしない
    ESCAPE = 'none'                        -- エスケープ文字: なし
    ESCAPE_UNENCLOSED_FIELD = '\134'     -- クォートなしフィールドのエスケープ文字
    DATE_FORMAT = 'auto'                   -- 日付形式: 自動判定
    TIMESTAMP_FORMAT = 'auto'             -- タイムスタンプ形式: 自動判定
    NULL_IF = ('')                         -- 空文字列を NULL として扱う
    COMMENT = 'file format for ingesting data for zero to snowflake';


In [ ]:
%%sql -r dataframe_6
-- ■ 作成したファイルフォーマットを確認する
-- SHOW コマンドで Snowflake に存在するオブジェクトの一覧と設定内容を確認できます。
SHOW FILE FORMATS IN DATABASE citibike;


### 4-3a｜Trips テーブルを作成

CitiBike の乗車データを格納するテーブルを作成します。

In [ ]:
%%sql -r dataframe_7
-- ■ CitiBike の乗車データを格納するテーブルを作成する
-- OR REPLACE を付けると既存テーブルがあっても再作成します。
-- 各カラムのデータ型を事前に定義しておくのがリレーショナルDBの基本です。
CREATE OR REPLACE TABLE trips (
    tripduration          INTEGER,   -- 乗車時間（秒）
    starttime             TIMESTAMP, -- 乗車開始日時
    stoptime              TIMESTAMP, -- 乗車終了日時
    start_station_id      INTEGER,   -- 乗車ステーション ID
    start_station_name    STRING,    -- 乗車ステーション名
    start_station_latitude  FLOAT,   -- 乗車ステーションの緯度
    start_station_longitude FLOAT,   -- 乗車ステーションの経度
    end_station_id        INTEGER,   -- 返却ステーション ID
    end_station_name      STRING,    -- 返却ステーション名
    end_station_latitude  FLOAT,     -- 返却ステーションの緯度
    end_station_longitude FLOAT,     -- 返却ステーションの経度
    bikeid                INTEGER,   -- 自転車 ID
    membership_type       STRING,    -- 会員種別
    usertype              STRING,    -- ユーザー種別（Subscriber / Customer）
    birth_year            INTEGER,   -- 生まれ年
    gender                INTEGER    -- 性別（0: 不明, 1: 男性, 2: 女性）
);


---
## Step 5｜データの読み込み（スケールアップを体感！）

> 💡 **Snowflake Tips: ウェアハウスのスケールアップ**
>
> Snowflake の **仮想ウェアハウス** は、クエリやデータロードのためのコンピューティングリソースです。
>
> | サイズ | vCPU 数（目安） | クレジット/時間 |
> |--------|---------------|----------------|
> | XS     | 1 クラスター   | 1              |
> | S      | 2 クラスター   | 2              |
> | M      | 4 クラスター   | 4              |
> | **L**  | **8 クラスター** | **8**        |
>
> - **1 サイズアップ ≈ 処理速度 2 倍（コストも 2 倍）**
> - 使用していない時は **自動サスペンド** でコスト節約可能
> - スケールアップ・ダウンは **数秒** で完了（再起動不要！）

まず S サイズでデータをロードし、その後 L サイズにスケールアップして速度の違いを体感します！

In [ ]:
%%sql -r dataframe_8
-- ■ 現在のウェアハウス設定を確認する
-- ウェアハウスのサイズ（XS/S/M/L など）や稼働状態を確認します。
-- LIKE 句でウェアハウス名を絞り込んでいます。
SHOW WAREHOUSES LIKE 'compute_wh';


### 5-2｜S サイズでデータロード

外部ステージから `trips` テーブルへデータをコピーします。  
> ⏱️ 実行時間を記録しておきましょう！（後で L サイズと比較します）

In [ ]:
%%sql -r dataframe_9
-- ■ ウェアハウスを S（Small）サイズに設定する
-- ALTER WAREHOUSE で既存ウェアハウスの設定を変更します。
-- small = 2 クラスター構成（クレジット消費: 2/時間）
ALTER WAREHOUSE compute_wh SET WAREHOUSE_SIZE = 'small';

-- ■ サイズが変更されたか確認する
SHOW WAREHOUSES LIKE 'compute_wh';


In [ ]:
%%sql -r dataframe_10
-- ■ 外部ステージから trips テーブルへデータをコピーする（S サイズ）
-- COPY INTO がデータロードの基本コマンドです。
-- ⏱️ 実行時間を記録しておきましょう！（Adaptive Warehouse と比較します）
COPY INTO trips          -- コピー先テーブル
FROM @citibike_trips     -- コピー元ステージ（@ でステージ参照）
FILE_FORMAT = csv        -- 使用するファイルフォーマット（先ほど作成したもの）
PATTERN = '.*csv.*';     -- 正規表現でロード対象ファイルを絞り込む（.csv を含むファイルのみ）


In [ ]:
%%sql -r dataframe_11
-- ■ テーブルのデータを全件削除する（テーブル構造は残す）
-- TRUNCATE = 全行削除。DELETE より高速で、テーブル定義（スキーマ）は保持されます。
-- ※ DELETE と異なり WHERE 句で絞り込めませんが、全件消す場合は TRUNCATE が高速です。
TRUNCATE TABLE trips;

-- ■ データが空になったことを確認する（0 件が返れば OK）
SELECT * FROM trips LIMIT 10; -- LIMIT で取得行数を制限（大量データの誤表示防止）


### 5-3｜Adaptive Warehouse に切り替え

> 💡 **Snowflake Tips: Adaptive Warehouse（Adaptive Compute）**
>
> **Adaptive Warehouse** は、ウェアハウスサイズを手動管理する必要がなくなる、**2026年6月 GA** の新機能です。
>
> | 従来のウェアハウス | Adaptive Warehouse |
> |-------------------|--------------------|
> | サイズ（XS〜6XL）を手動で選ぶ | サイズ設定不要。自動でリソース最適化 |
> | マルチクラスター設定が必要 | 自動スケール（設定不要） |
> | Query Acceleration Service の設定 | 自動で組み込み済み |
> | 停止・再開ポリシーの管理 | 管理不要 |
>
> **主な設定パラメータ:**
> - `MAX_QUERY_PERFORMANCE_LEVEL`：1クエリに使える最大パフォーマンス上限（デフォルト: `XLARGE`）
> - `QUERY_THROUGHPUT_MULTIPLIER`：同時実行量の倍率（デフォルト: `2`）
>
> **課金モデル:** クエリ単位の従量課金（ウェアハウス作成時は課金なし。最初のクエリ実行から課金開始）
>
> ⚠️ Enterprise Edition 以上が必要。

In [ ]:
-- ■ Adaptive Warehouse を新規作成する
-- 従来のウェアハウスはサイズ（XS〜6XL）を手動で選ぶ必要がありましたが、
-- Adaptive Warehouse はクエリの内容に応じてリソースを自動で最適化します。
CREATE OR REPLACE ADAPTIVE WAREHOUSE analytics_wh
    WITH
    -- 1 クエリあたりの最大パフォーマンス上限（XS/S/M/L/XL/XXL/XXXL/X4L から選択）
    -- システムが高い確信を持って最適化できる場合にこの上限まで使います。
    MAX_QUERY_PERFORMANCE_LEVEL = LARGE
    -- 同時実行のスループット倍率（デフォルト 2）
    -- 値を上げると同時実行数が増えてキュー待ちが減りますが、コストが上がる可能性があります。
    QUERY_THROUGHPUT_MULTIPLIER = 2;

-- ■ 作成した Adaptive Warehouse を確認する
-- STATE / MAX_QUERY_PERFORMANCE_LEVEL カラムに注目！
-- WAREHOUSE_SIZE は NULL になります（サイズ管理が不要なため）。
SHOW WAREHOUSES LIKE 'analytics_wh';


### 5-4｜Adaptive Warehouse でデータロード

> ⏱️ S サイズと比較して、Adaptive Warehouse は **リソースを自動最適化** するため  
> 大量データロードでも効率よく実行できます。

In [ ]:
%%sql -r dataframe_13
-- ■ Adaptive Warehouse に切り替える
-- USE WAREHOUSE でクエリ実行に使うウェアハウスを切り替えます。
USE WAREHOUSE analytics_wh;

-- ■ Adaptive Warehouse でデータロードを実行する（S サイズと速度を比較！）
-- ⏱️ 実行時間を S サイズと比べてみましょう。
-- Adaptive Warehouse はデータロードの並列度も自動で最適化します。
COPY INTO trips          -- コピー先テーブル
FROM @citibike_trips     -- コピー元ステージ
FILE_FORMAT = csv        -- ファイルフォーマット
PATTERN = '.*csv.*';     -- .csv を含むファイルのみ対象


In [ ]:
%%sql -r dataframe_14
-- ■ 以降のステップで使う標準ウェアハウスに戻す
USE WAREHOUSE compute_wh;

-- ■【参考】既存の標準ウェアハウスを Adaptive Warehouse に変換する場合
-- コンバート中もクエリは継続実行されます（ダウンタイムなし）。
-- ALTER WAREHOUSE compute_wh SET WAREHOUSE_TYPE = 'ADAPTIVE';

-- ■【参考】Adaptive Warehouse を標準ウェアハウスに戻す場合
-- ALTER WAREHOUSE analytics_wh SET WAREHOUSE_TYPE = 'STANDARD';

-- analytics_wh は Step 11（クリーンアップ）で削除します。


---
## Step 6｜クエリ・リザルトキャッシュ・クローンの操作

> 💡 **Snowflake Tips: 3 種類のキャッシュ**
>
> Snowflake には **3 種類のキャッシュ** があり、クエリを高速化・コスト削減します。
>
> | キャッシュ | 説明 | 有効期間 |
> |-----------|------|----------|
> | **メタデータキャッシュ** | `COUNT(*)` など Cloud Service Layer で完結 | 常時 |
> | **クエリリザルトキャッシュ** | 同じ SQL の実行結果を再利用 | 24 時間 |
> | **データキャッシュ** | ウェアハウスの SSD にスキャン済みデータを保持 | WH 稼働中 |
>
> クエリプロファイル（⛅ マーク）で各キャッシュの使用状況を確認できます！

### 6-1｜trips テーブルの内容確認

In [ ]:
%%sql -r dataframe_15
-- ■ trips テーブルの中身を確認する
-- SELECT * で全カラムを取得します。
-- LIMIT 20 で先頭 20 行だけ取得（大量データの誤全件表示を防ぐために必ず付けましょう）。
SELECT * FROM trips LIMIT 20;


### 6-2｜2017 年下半期以降の集計クエリ

> 💡 **マイクロパーティション & プルーニング**  
> Snowflake はデータを **マイクロパーティション**（50〜500MB の圧縮ブロック）で管理します。  
> `WHERE` 条件を付けることで不要なパーティションをスキップ（**プルーニング**）でき、  
> スキャン量を大幅に削減できます。クエリプロファイルで確認してみましょう！

In [ ]:
%%sql -r dataframe_16
-- ■ 2017 年下半期以降の月 × 性別ごとの利用回数を集計する
-- WHERE で条件を絞ることで Snowflake のパーティションプルーニングが効き、
-- スキャンするデータ量が減って高速になります（クエリプロファイルで確認！）。
SELECT
    monthname(starttime) AS month,  -- starttime から月名（Jan/Feb...）を取得
    gender,                          -- 性別（0:不明, 1:男性, 2:女性）
    count(*) AS num_trips            -- 該当グループの行数（乗車回数）
FROM trips
WHERE starttime > '2017-07-01 00:00:00' -- 2017 年 7 月以降のデータに絞り込む
GROUP BY monthname(starttime), gender   -- 月名 × 性別 でグループ化
ORDER BY 1, 2;                          -- 1 列目（月名）→ 2 列目（性別）の順にソート


### 6-3｜メタデータキャッシュ体験

> 💡 **メタデータキャッシュ**  
> `COUNT(*)` は Snowflake のメタデータから直接取得できます。  
> ウェアハウスを起動せず **ほぼ瞬時に結果が返る** のが特徴です！  
> クエリプロファイルで「Cloud Service Layer」で完結していることを確認しましょう。

In [ ]:
%%sql -r dataframe_17
-- ■ テーブルの総行数を確認する（メタデータキャッシュ体験）
-- COUNT(*) は Snowflake のメタデータから直接取得できるため、
-- ウェアハウスを起動せずほぼ瞬時に結果が返ります！
-- クエリプロファイルで「Cloud Service Layer」だけで完結していることを確認しましょう。
SELECT count(*) FROM trips;


### 6-4｜クエリリザルトキャッシュ体験

> 💡 **クエリリザルトキャッシュ**  
> **まったく同じ SQL** を 24 時間以内に再実行すると、キャッシュされた結果が即時返却されます。  
> データに変更がなければウェアハウス不要→**クレジット消費なし**！  
> クエリプロファイルで `QUERY RESULT REUSE` が表示されることを確認しましょう。

以下の **全く同じクエリを 2 回実行** して実行時間を比較します。

In [ ]:
%%sql -r dataframe_18
-- ■ 1 時間ごとの基本統計量を集計する（1 回目の実行）
-- ⏱️ 実行時間を記録しておきましょう！（2 回目と比較します）
SELECT
    date_trunc('hour', starttime)  AS "date",             -- starttime を時間単位で丸める
    count(*)                       AS "num trips",        -- 1 時間あたりの乗車回数
    avg(tripduration) / 60         AS "avg duration (mins)", -- 平均乗車時間（秒 → 分に変換）
    avg(haversine(                                         -- haversine = 球面距離計算関数
        start_station_latitude, start_station_longitude,  -- 乗車地点の緯度・経度
        end_station_latitude, end_station_longitude)      -- 返却地点の緯度・経度
    )                              AS "avg distance (km)" -- 平均移動距離（km）
FROM trips
GROUP BY 1   -- 第 1 カラム（date_trunc の結果）でグループ化
ORDER BY 1;  -- 日時順に昇順ソート


In [ ]:
%%sql -r dataframe_19
-- ■ まったく同じクエリを 2 回目実行する（クエリリザルトキャッシュ体験）
-- ⏱️ 1 回目と比べてほぼ瞬時に結果が返るはずです！
-- クエリプロファイルで「QUERY RESULT REUSE」が表示されることを確認しましょう。
-- 同一 SQL・同一データなら 24 時間以内はキャッシュされた結果を再利用します（ウェアハウス不要！）。
SELECT
    date_trunc('hour', starttime)  AS "date",
    count(*)                       AS "num trips",
    avg(tripduration) / 60         AS "avg duration (mins)",
    avg(haversine(
        start_station_latitude, start_station_longitude,
        end_station_latitude, end_station_longitude)
    )                              AS "avg distance (km)"
FROM trips
GROUP BY 1
ORDER BY 1;


### 6-5｜データキャッシュ体験

> 💡 **データキャッシュ（ローカルディスクキャッシュ）**  
> ウェアハウスがスキャンしたデータは SSD にキャッシュされます。  
> 類似クエリを連続実行すると、2 回目以降はキャッシュ済みデータを使用してスキャン量が減ります。  
> クエリプロファイルで `% scanned from cache` の値を確認しましょう！

条件を少し変えた 2 つのクエリで、データキャッシュの効果を確認します。

In [ ]:
%%sql -r dataframe_20
-- ■ 2018 年以降のデータを集計する（1 回目：ウェアハウスがデータをスキャン）
-- ⏱️ このクエリ実行後、ウェアハウスの SSD にスキャン済みデータがキャッシュされます。
SELECT
    monthname(starttime) AS month, -- 月名
    gender,                         -- 性別
    count(*) AS num_trips           -- 乗車回数
FROM trips
WHERE starttime > '2018-01-01 00:00:00' -- 2018 年以降に絞り込む
GROUP BY monthname(starttime), gender;


In [ ]:
%%sql -r dataframe_21
-- ■ 2017 年以降のデータを集計する（2 回目：データキャッシュが効く）
-- 1 回目より広い範囲ですが、2018 年以降の部分はキャッシュ済みデータを再利用します。
-- クエリプロファイルで「% scanned from cache」の値を確認しましょう！
-- 高い割合ほどキャッシュが効いていて、実際のディスク読み込みが少ないことを意味します。
SELECT
    monthname(starttime) AS month,
    gender,
    count(*) AS num_trips
FROM trips
WHERE starttime > '2017-01-01 00:00:00' -- 2017 年以降（1 回目より範囲が広い）
GROUP BY monthname(starttime), gender;


---
## Step 7｜半構造化データ・ビュー・結合の操作

6-5 で季節性があることがわかりました。**天気との相関** も見てみましょう！

> 💡 **Snowflake Tips: VARIANT 型（半構造化データ）**
>
> Snowflake の **VARIANT 型** は JSON、Avro、ORC、Parquet、XML などの  
> **半構造化データをそのまま格納できる** 特殊な型です。
>
> ```sql
> -- JSON の値にアクセス
> v:station::string        -- キー名でアクセス + 型キャスト
> v:weatherCondition       -- キャストなしも可（VARIANT のまま）
> v:observations[0]:temp   -- 配列要素へのアクセス
> ```
>
> スキーマを事前定義せず **ELT**（Extract → Load → Transform）が実現できます！

In [ ]:
%%sql -r dataframe_22
-- ■ Weather データベースを作成する
-- IF NOT EXISTS を付けると既にある場合はエラーにならず安全に実行できます。
CREATE DATABASE IF NOT EXISTS weather;

-- ■ 以降の作業のためロール・ウェアハウス・データベース・スキーマを切り替える
USE ROLE sysadmin;        -- オブジェクト作成権限を持つロールに切り替え
USE WAREHOUSE compute_wh; -- 計算リソース（ウェアハウス）を指定
USE DATABASE weather;     -- 操作対象のデータベースを切り替え
USE SCHEMA public;        -- 操作対象のスキーマを指定


In [ ]:
%%sql -r dataframe_23
-- ■ JSON データをそのまま格納できるテーブルを作成する
-- VARIANT 型のカラム 1 つだけ持つシンプルな構造です。
-- VARIANT 型 = JSON / Avro / Parquet など半構造化データを格納できる Snowflake 独自の型。
-- スキーマ（カラム定義）を事前に決めなくてよいため、柔軟な ELT が実現できます。
CREATE OR REPLACE TABLE json_weather_data (v VARIANT);


In [ ]:
%%sql -r dataframe_24
-- ■ Weather データ用の外部ステージを作成する
-- citibike と同様に、AWS S3 上のデータを参照する外部ステージを定義します。
-- 今回は JSON 形式の気象データが格納されています。
CREATE STAGE nyc_weather
    url = 's3://snowflake-workshop-lab/zero-weather-nyc';


In [ ]:
%%sql -r dataframe_25
-- ■ Weather ステージ内のファイル一覧を確認する
-- どのようなファイルが S3 に格納されているかを確認します。
LIST @nyc_weather;


In [ ]:
%%sql -r dataframe_26
-- ■ JSON データをテーブルにロードする
-- FILE_FORMAT をインラインで指定しています（事前作成済みのオブジェクトを指定してもOK）。
-- STRIP_OUTER_ARRAY = true：JSON の最外殻が配列（[ ]）の場合、配列を展開して 1 行ずつ格納。
COPY INTO json_weather_data
FROM @nyc_weather
    FILE_FORMAT = (TYPE = json STRIP_OUTER_ARRAY = true);


In [ ]:
%%sql -r dataframe_27
-- ■ ロードした JSON データを確認する
-- VARIANT 型のカラム v にはネストされた JSON がそのまま入っています。
-- v:station のように「v:[キー名]」の記法で JSON の値にアクセスできます。
SELECT * FROM json_weather_data LIMIT 10;


### 7-5｜JSON データをビューで構造化

VARIANT 型の JSON カラムに対して、`v:[key名]::型` の記法で各フィールドを取り出し、  
通常のリレーショナルテーブルのように扱えるビューを作成します。

In [ ]:
%%sql -r dataframe_28
-- ■ JSON（VARIANT 型）を通常テーブルのように扱えるビューを作成する
-- v:[キー名]::型 の記法で JSON のフィールドを取り出し、型を指定します。
-- ビューを使うことで、利用者は JSON 記法を意識せずに通常の SQL で分析できます。
CREATE OR REPLACE VIEW json_weather_data_view AS
SELECT
    v:obsTime::timestamp        AS observation_time,       -- 観測日時（timestamp 型にキャスト）
    v:station::string           AS station_id,             -- 観測ステーション ID
    v:name::string              AS city_name,              -- 都市名
    v:country::string           AS country,                -- 国コード
    v:latitude::float           AS city_lat,               -- 緯度
    v:longitude::float          AS city_lon,               -- 経度
    v:weatherCondition::string  AS weather_conditions,     -- 天気状況（晴れ・雨・曇りなど）
    v:coco::int                 AS weather_conditions_code,-- 天気コード（数値）
    v:temp::float               AS temp,                   -- 気温（℃）
    v:prcp::float               AS rain,                   -- 降水量（mm）
    v:tsun::float               AS tsun,                   -- 日照時間（分）
    v:wdir::float               AS wind_dir,               -- 風向き（度）
    v:wspd::float               AS wind_speed,             -- 風速（km/h）
    v:dwpt::float               AS dew_point,              -- 露点温度（℃）
    v:rhum::float               AS relative_humidity,      -- 相対湿度（%）
    v:pres::float               AS pressure                -- 気圧（hPa）
FROM
    json_weather_data
WHERE
    station_id = '72502'; -- ニューヨーク（JFK 空港付近）の観測ステーションに絞る


In [ ]:
%%sql -r dataframe_29
-- ■ 作成したビューのデータを確認する
-- ビューを通すことで、JSON だったデータが通常のカラム形式で表示されます。
-- date_trunc('month', ...) で 2018 年 1 月のデータに絞り込んでいます。
SELECT * FROM json_weather_data_view
WHERE date_trunc('month', observation_time) = '2018-01-01' -- 2018 年 1 月のみ抽出
LIMIT 20;


### 7-6｜Trips データと Weather データを結合

CitiBike の乗車データと気象データを **1 時間単位** で JOIN して、  
天気条件別の利用数を分析します！

In [ ]:
%%sql -r dataframe_30
-- ■ CitiBike の乗車データと気象データを時刻で結合して天気別乗車回数を集計する
-- date_trunc('hour', ...) で両テーブルを「1 時間単位」に丸めてから JOIN します。
-- LEFT OUTER JOIN = 左テーブル（trips）の全行を残し、一致する天気データがない行も含めます。
SELECT
    weather_conditions  AS conditions, -- 天気条件（晴れ・雨・曇りなど）
    count(*)            AS num_trips   -- その天気のときの乗車回数
FROM citibike.public.trips                 -- 乗車データ（完全修飾名でデータベースを指定）
LEFT OUTER JOIN json_weather_data_view     -- 気象データビューと結合
    ON date_trunc('hour', observation_time) = date_trunc('hour', starttime) -- 1 時間単位で結合
WHERE conditions IS NOT NULL  -- 天気データが取得できた行のみ集計
GROUP BY 1                    -- 天気条件でグループ化
ORDER BY 2 DESC;              -- 乗車回数の多い順にソート
-- アメリカ人は晴れの日でも雨の日でも自転車を使う？！結果を確認してみましょう！


---
## Step 8｜タイムトラベル・ゼロコピークローン

> 💡 **Snowflake Tips: Time Travel（タイムトラベル）**
>
> Snowflake は変更前のデータを **自動的に保持** しており、過去の状態に戻れます。
>
> | 機能 | コマンド例 |
> |------|----------|
> | 削除テーブルの復元 | `UNDROP TABLE <name>` |
> | 過去時点のデータ参照 | `SELECT ... AT (TIMESTAMP => ...)` |
> | 特定 SQL 実行前の状態 | `SELECT ... BEFORE (STATEMENT => 'query_id')` |
>
> - Standard Edition: **最大 1 日**
> - Enterprise Edition: **最大 90 日**

### 8-1｜テーブルの誤削除 → Undrop で復元

In [ ]:
%%sql -r dataframe_31
-- ■ データベース・スキーマを Weather に切り替える
USE DATABASE weather;
USE SCHEMA public;

-- ■ (誤って...) json_weather_data テーブルを削除する
-- 通常 DROP TABLE は「元に戻せない」操作ですが、Snowflake では Time Travel で復元できます！
DROP TABLE json_weather_data;
-- ⚠️ テーブルが削除されました。次のセルで確認してみましょう。


In [ ]:
%%sql -r dataframe_32
-- ■ テーブルが存在しないことを確認する
-- 削除したテーブルを SELECT すると「Table not found」エラーが発生します。
-- ⚠️ エラーが出るのが正常です！このエラーを見て次の UNDROP を体験します。
SELECT * FROM json_weather_data LIMIT 10;


In [ ]:
%%sql -r dataframe_33
-- ■ UNDROP でテーブルを復元する（Time Travel の機能）
-- Snowflake は DROP 後もデータを保持しているため、UNDROP で即座に復元できます。
-- 復元可能な期間: Standard Edition = 最大 1 日 / Enterprise Edition = 最大 90 日
UNDROP TABLE json_weather_data;


In [ ]:
%%sql -r dataframe_34
-- ■ テーブルが復元されたことを確認する
-- UNDROP 後に SELECT できれば、データが正しく復元されています。
SELECT * FROM json_weather_data LIMIT 10;


### 8-2｜誤 UPDATE のロールバック

今度は **全ステーション名を 'oops' に変更** してしまった失敗を  
タイムトラベルで復元します！

In [ ]:
%%sql -r dataframe_35
-- ■ Time Travel（DML 復元）のために citibike データベースに切り替える
USE ROLE sysadmin;        -- オブジェクト操作権限を持つロールに切り替え
USE WAREHOUSE compute_wh; -- 使用するウェアハウスを指定
USE DATABASE citibike;    -- 操作対象のデータベースを切り替え
USE SCHEMA public;        -- 操作対象のスキーマを指定


In [ ]:
%%sql -r dataframe_36
-- ■ (意図的に誤って...) 全ステーション名を 'oops' に上書きする
-- UPDATE は WHERE なしだと全行を更新します。実務では絶対にやってはいけない操作です！
-- ここでは Time Travel の体験のために、あえて意図的にやっています。
UPDATE trips SET start_station_name = 'oops';
-- ⚠️ 約 6,150 万行のステーション名が全て 'oops' に変わってしまいました...


In [ ]:
%%sql -r dataframe_37
-- ■ UPDATE の影響を確認する（全行が 'oops' になっていることを確認）
-- 乗車回数の多いステーション上位 20 件を表示しますが、全て 'oops' が返るはずです。
SELECT
    start_station_name AS "station", -- ステーション名（全て 'oops' のはず...）
    count(*)           AS "rides"    -- そのステーションからの乗車回数
FROM trips
GROUP BY 1        -- ステーション名でグループ化
ORDER BY 2 DESC   -- 乗車回数の多い順
LIMIT 20;         -- 上位 20 件


タイムトラベルで **UPDATE 前の状態** に戻します。  
`QUERY_HISTORY` から直前の UPDATE の `query_id` を取得し、  
`BEFORE (STATEMENT => query_id)` でその実行前の状態のテーブルを再作成します。

In [ ]:
%%sql -r dataframe_38
-- ■ STEP 1: 直前の UPDATE のクエリ ID をセッション変数に格納する
-- SET で変数を定義し、サブクエリで Query History から該当クエリを探します。
-- INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION = 現セッションのクエリ履歴を返すテーブル関数
SET query_id = (
    SELECT query_id   -- クエリを一意に識別する ID
    FROM TABLE(information_schema.query_history_by_session(result_limit => 20))
    WHERE query_text ILIKE 'update%' -- SQL が 'update' で始まるクエリを検索（大文字小文字無視）
    ORDER BY start_time DESC         -- 最新のものから並べる
    LIMIT 1                          -- 最も直近の 1 件だけ取得
);

-- ■ STEP 2: 取得したクエリ ID を確認する（$ でセッション変数を参照）
SELECT $query_id;

-- ■ STEP 3: タイムトラベルで UPDATE 実行前の状態に戻してテーブルを再作成する
-- BEFORE (STATEMENT => ...) = 指定したクエリ実行「直前」の状態のデータを参照します。
-- これが Snowflake の Time Travel（DML ロールバック）の核心です！
CREATE OR REPLACE TABLE trips AS
(SELECT * FROM trips BEFORE (STATEMENT => $query_id));


In [ ]:
%%sql -r dataframe_39
-- ■ ステーション名が復元されているか確認する
-- 複数のステーション名が返ってくれば Time Travel による復元成功です！
SELECT
    start_station_name AS "station", -- ステーション名（正常な値が返るはず）
    count(*)           AS "rides"    -- 乗車回数
FROM trips
GROUP BY 1
ORDER BY 2 DESC -- 乗車回数の多い順
LIMIT 20;


### 8-0｜ゼロコピークローン

> 💡 **Snowflake Tips: Zero-Copy Clone（ゼロコピークローン）**
>
> Snowflake のクローンは **ポインターのコピー** です。物理的なデータのコピーは発生しません！
>
> - クローン作成は **ほぼ瞬時**（データ量に関係なし）
> - クローン後に変更が生じた分だけストレージが増加
> - テーブル、スキーマ、データベース全体もクローン可能
> - 本番テーブルを直接触らず **開発・テスト用クローン** を作るのがベストプラクティス！

In [ ]:
%%sql -r dataframe_40
-- ■ trips テーブルのゼロコピークローンを作成する
-- CLONE = メタデータ（ポインター）のみコピーするため、物理データのコピーは発生しません。
-- 何億行あっても一瞬で複製でき、ストレージコストも増えません！
-- 本番テーブル（trips）を直接変更せず、開発用クローン（trips_dev）で安全に作業します。
CREATE TABLE trips_dev CLONE trips;


---
## Step 9｜ロール・権限管理（RBAC）

> 💡 **Snowflake Tips: ロールベースアクセス制御（RBAC）**
>
> Snowflake のアクセス制御は **ロール（Role）** ベースです。
>
> **デフォルトのロール階層:**
> ```
> ACCOUNTADMIN（最上位）
>   └── SYSADMIN（オブジェクト作成・管理）
>         └── PUBLIC（全ユーザーが持つ最低限の権限）
> ```
>
> **ベストプラクティス:**
> - `ACCOUNTADMIN` は日常業務では使わず、必要時のみ使用
> - 用途別にカスタムロールを作成する
> - **最小権限の原則** を遵守する

ここでは `junior_dba` という新しいロールを作成し、権限を段階的に付与します。

In [ ]:
%%sql -r dataframe_41
-- ■ ACCOUNTADMIN ロールに切り替える
-- ACCOUNTADMIN = Snowflake で最上位の権限を持つロール。ロールの作成・付与ができます。
USE ROLE accountadmin;

-- ■ 新しいロール junior_dba を作成する
-- CREATE ROLE でカスタムロールを自由に作成できます。
CREATE ROLE junior_dba;

-- ■ 作成したロールを自分のユーザーに割り当てる
-- GRANT ROLE でロールをユーザーに付与します。
-- これにより junior_dba ロールで操作できるようになります。
GRANT ROLE junior_dba TO USER ysakuragi;


In [ ]:
%%sql -r dataframe_42
-- ■ junior_dba ロールに切り替えてみる
USE ROLE junior_dba;

-- ■ この状態でウェアハウスやデータベースを使おうとするとエラーになります。
-- まだウェアハウスや DB への使用権限（USAGE）が付与されていないためです。
-- 次のセルで権限を段階的に付与していきます。


In [ ]:
%%sql -r dataframe_43
-- ■ ACCOUNTADMIN に戻って compute_wh の使用権限を junior_dba に付与する
-- GRANT USAGE ON WAREHOUSE = そのウェアハウスを「使う」権限（起動・クエリ実行）を付与します。
USE ROLE accountadmin;
GRANT USAGE ON WAREHOUSE compute_wh TO ROLE junior_dba;


In [ ]:
%%sql -r dataframe_44
-- ■ citibike と weather データベースの使用権限を junior_dba に付与する
-- GRANT USAGE ON DATABASE = そのデータベースの存在を認識し、中のオブジェクトにアクセスする権限。
-- ※ USAGE だけではテーブルの SELECT はできません。テーブルにも GRANT SELECT が必要です。
GRANT USAGE ON DATABASE citibike TO ROLE junior_dba;
GRANT USAGE ON DATABASE weather  TO ROLE junior_dba;


In [ ]:
%%sql -r dataframe_45
-- ■ junior_dba ロールで実際に操作できるか確認する
USE ROLE junior_dba;
USE WAREHOUSE compute_wh; -- ウェアハウスを使えるようになったか確認

-- ■ アクセス可能なデータベースの一覧を表示する
-- USAGE 権限を付与したデータベースが表示されれば成功です！
SHOW DATABASES;


---
## Step 11｜Snowflake 環境のリセット

> ⚠️ **注意**: 以下のセルを実行すると、このハンズオンで作成した **全オブジェクトが削除** されます。  
> ハンズオン終了後に実行してください。

In [ ]:
%%sql -r dataframe_46
-- ■ ハンズオンで作成した全オブジェクトを削除する
-- ACCOUNTADMIN ロールで DROP を実行します。
USE ROLE accountadmin;

-- IF EXISTS を付けると、対象が存在しない場合もエラーにならず安全に実行できます。
DROP SHARE     IF EXISTS zero_to_snowflake_shared_data; -- 共有（作成した場合）
DROP DATABASE  IF EXISTS citibike;    -- CitiBike データベース（中のテーブル・ステージ等も削除）
DROP DATABASE  IF EXISTS weather;     -- Weather データベース
DROP WAREHOUSE IF EXISTS analytics_wh;-- Adaptive Warehouse
DROP ROLE      IF EXISTS junior_dba;  -- カスタムロール

-- ■ クエリリザルトキャッシュの設定を元に戻す
ALTER ACCOUNT SET USE_CACHED_RESULT = false; -- 一度無効化して
ALTER ACCOUNT SET USE_CACHED_RESULT = true;  -- 再度有効化（デフォルト状態に戻す）


---
## 🎉 おつかれさまでした！

このハンズオンでは以下の Snowflake 機能を体験しました：

- ✅ データベース・スキーマ・ウェアハウスの作成
- ✅ 外部ステージからの CSV データロード
- ✅ ウェアハウスのスケールアップによるパフォーマンス向上
- ✅ 3 種類のキャッシュ（メタデータ・リザルト・データ）
- ✅ VARIANT 型による半構造化データ（JSON）のロードと操作
- ✅ タイムトラベルによるデータ復元
- ✅ ゼロコピークローンによる即座のテーブル複製
- ✅ ロールベースアクセス制御（RBAC）

### 次のステップ 🚀

- [Snowflake ドキュメント](https://docs.snowflake.com/ja/) で詳細を学ぶ
- Marketplace でサンプルデータを試す
- Cortex AI 機能（`AI_COMPLETE`, `CORTEX SEARCH` など）を試す